In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.read_csv("../data/pseudo_labeled_tickets.csv")

df.head()

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,...,Keyword_Score,Resolution_Score,Semantic_Score,Final_Severity_Score,Keyword_Score_Normalized,Resolution_Score_Normalized,Inferred_Severity,Mismatch_Label,Mismatch_Type,Severity_Delta
0,TKT-100000,George Simon,lisastrickland@example.com,Hours of operation - Individual,"Hi Support, Where is your headquarters located...",General Inquiry,High,Web Form,2025-07-02,43,...,0,2,0.593736,0.430201,0.000000,0.666667,Medium,0,Consistent,-1
1,TKT-100001,Scott Thompson,wevans@example.org,Data not syncing - Card,"Hi Support, The application crashes every time...",Technical,High,Chat,2025-06-28,41,...,0,2,0.173507,0.220087,0.000000,0.666667,Low,1,False Alarm,-2
2,TKT-100002,Jennifer Smith,oleonard@example.net,2FA issues - Question,"Hi Support, How do I upgrade to the Enterprise...",Account,High,Web Form,2025-02-05,7,...,0,3,0.588986,0.494493,0.000000,1.000000,High,0,Consistent,0
3,TKT-100003,Rachel Bullock,katherine67@example.net,Login failed - Let,"Hi Support, The dashboard is not loading any d...",Technical,Low,Web Form,2025-03-20,41,...,4,2,0.095357,0.352440,0.571429,0.666667,Low,0,Consistent,0
4,TKT-100004,Thomas Parks DDS,raykelsey@example.com,Refund status - Attention,"Hi Support, I have been trying to update my pa...",Billing,Medium,Email,2025-04-27,40,...,0,2,0.649460,0.458063,0.000000,0.666667,Medium,0,Consistent,0


In [3]:
priority_map = {
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}

In [4]:
severity_keywords = {
    "outage":5,
    "fraud":5,
    "hacked":5,
    "stolen":5,
    "payment failed":4,
    "unauthorized":4,
    "error":3,
    "failed":3,
    "urgent":2,
    "asap":1,
    "login":1,
    "password":1
}

In [5]:
def extract_keywords(text):

    text = str(text).lower()

    found = []

    for word in severity_keywords:

        if word in text:
            found.append(word)

    return found

In [6]:
df["Detected_Keywords"] = (
    df["Combined_Text"]
    .apply(extract_keywords)
)

In [7]:
df[
[
"Combined_Text",
"Detected_Keywords"
]
].head()

,Combined_Text,Detected_Keywords
0,"Hours of operation - Individual Hi Support, Wh...",[]
1,"Data not syncing - Card Hi Support, The applic...",[]
2,"2FA issues - Question Hi Support, How do I upg...",[]
3,"Login failed - Let Hi Support, The dashboard i...","[failed, login]"
4,"Refund status - Attention Hi Support, I have b...",[]


In [8]:
def generate_analysis(row):

    if row["Mismatch_Type"] == "Hidden Crisis":

        return (
            "The ticket exhibits semantic and "
            "contextual indicators of high urgency "
            "that are inconsistent with the assigned "
            "priority level."
        )

    elif row["Mismatch_Type"] == "False Alarm":

        return (
            "The assigned priority appears higher "
            "than suggested by the semantic and "
            "operational evidence extracted from "
            "the ticket."
        )

    else:

        return (
            "The inferred severity is consistent "
            "with the assigned priority."
        )

In [9]:
def confidence(row):

    delta = abs(row["Severity_Delta"])

    score = 70 + delta * 10

    if row["Keyword_Score"] >= 5:
        score += 5

    return min(score, 99)

In [10]:
def generate_dossier(row):

    dossier = {

        "ticket_id":
        row["Ticket_ID"],

        "assigned_priority":
        row["Priority_Level"],

        "inferred_severity":
        row["Inferred_Severity"],

        "mismatch_type":
        row["Mismatch_Type"],

        "severity_delta":
        int(row["Severity_Delta"]),

        "feature_evidence":[

            {
                "signal":"keyword",
                "value":
                row["Detected_Keywords"]
            },

            {
                "signal":"resolution_time",

                "value":
                int(
                    row[
                    "Resolution_Time_Hours"
                    ]
                ),

                "interpretation":

                (
                "Fast resolution suggests "
                "operational urgency"

                if row[
                "Resolution_Time_Hours"
                ] <= 12

                else

                "Normal resolution window"

                )

            },

            {

                "signal":
                "semantic_score",

                "value":

                round(
                row[
                "Semantic_Score"
                ],
                3
                )

            }

        ],

        "constraint_analysis":

        generate_analysis(
        row
        ),

        "confidence":

        str(
        confidence(
        row
        )
        ) + "%"

    }

    return dossier

In [11]:
sample = df.iloc[0]

dossier = generate_dossier(
    sample
)

print(
    json.dumps(
        dossier,
        indent=4
    )
)

{
    "ticket_id": "TKT-100000",
    "assigned_priority": "High",
    "inferred_severity": "Medium",
    "mismatch_type": "Consistent",
    "severity_delta": -1,
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": []
        },
        {
            "signal": "resolution_time",
            "value": 43,
            "interpretation": "Normal resolution window"
        },
        {
            "signal": "semantic_score",
            "value": 0.594
        }
    ],
    "constraint_analysis": "The inferred severity is consistent with the assigned priority.",
    "confidence": "80%"
}


In [12]:
flagged = df[
    df["Mismatch_Label"] == 1
]

flagged.shape

(5441, 24)

In [13]:
dossiers = []

for _, row in flagged.iterrows():

    dossiers.append(

        generate_dossier(
            row
        )

    )

In [14]:
with open(
    "../data/evidence_dossiers.json",
    "w"
) as f:

    json.dump(
        dossiers,
        f,
        indent=4
    )

In [15]:
hidden = df[
    df[
    "Mismatch_Type"
    ] == "Hidden Crisis"
].iloc[0]

print(

json.dumps(

generate_dossier(
hidden
),

indent=4

)

)

{
    "ticket_id": "TKT-100007",
    "assigned_priority": "Medium",
    "inferred_severity": "Critical",
    "mismatch_type": "Hidden Crisis",
    "severity_delta": 2,
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": [
                "payment failed",
                "failed"
            ]
        },
        {
            "signal": "resolution_time",
            "value": 27,
            "interpretation": "Normal resolution window"
        },
        {
            "signal": "semantic_score",
            "value": 0.634
        }
    ],
    "constraint_analysis": "The ticket exhibits semantic and contextual indicators of high urgency that are inconsistent with the assigned priority level.",
    "confidence": "95%"
}


In [16]:
false_alarm = df[
    df[
    "Mismatch_Type"
    ] == "False Alarm"
].iloc[0]

print(

json.dumps(

generate_dossier(
false_alarm
),

indent=4

)

)

{
    "ticket_id": "TKT-100001",
    "assigned_priority": "High",
    "inferred_severity": "Low",
    "mismatch_type": "False Alarm",
    "severity_delta": -2,
    "feature_evidence": [
        {
            "signal": "keyword",
            "value": []
        },
        {
            "signal": "resolution_time",
            "value": 41,
            "interpretation": "Normal resolution window"
        },
        {
            "signal": "semantic_score",
            "value": 0.174
        }
    ],
    "constraint_analysis": "The assigned priority appears higher than suggested by the semantic and operational evidence extracted from the ticket.",
    "confidence": "90%"
}


In [17]:
{
    "signal": "keyword",
    "value": [
        "payment failed",
        "failed"
    ],
    "weight": 0.83
}

{'signal': 'keyword', 'value': ['payment failed', 'failed'], 'weight': 0.83}

In [18]:
{
    "signal":"resolution_time",
    "value":27,
    "interpretation":"Normal resolution window",
    "weight":0.35
}

{'signal': 'resolution_time',
 'value': 27,
 'interpretation': 'Normal resolution window',
 'weight': 0.35}

In [19]:
{
    "signal":"semantic_score",
    "value":0.634,
    "weight":0.76
}

{'signal': 'semantic_score', 'value': 0.634, 'weight': 0.76}

In [20]:
{
    "signal":"keyword",

    "value":
    row["Detected_Keywords"],

    "weight":

    round(
        row["Keyword_Score"] /
        df["Keyword_Score"].max(),
        2
    )
}

{'signal': 'keyword', 'value': [], 'weight': 0.0}

In [21]:
def generate_analysis(row):

    keywords = ", ".join(
        row["Detected_Keywords"]
    )

    if row["Mismatch_Type"] == "Hidden Crisis":

        return (

            f"The presence of "
            f"{keywords} together with a "
            f"semantic urgency score of "
            f"{row['Semantic_Score']:.2f} "
            f"suggests that the ticket "
            f"deserves a higher priority "
            f"than the assigned "
            f"{row['Priority_Level']} level."

        )

    elif row["Mismatch_Type"] == "False Alarm":

        return (

            f"The ticket content and "
            f"semantic indicators do not "
            f"strongly support the assigned "
            f"{row['Priority_Level']} priority."

        )

    return (

        "The inferred severity agrees "
        "with the assigned priority."

    )